In [ ]:
# install the neede libraries
%pip install pyiqa
%pip install pandas
%pip install scikit-learn
%pip install tqdm

In [ ]:
# import the needed libraries
import os
import numpy as np
import pandas as pd
import torch
import pyiqa
from tqdm import tqdm
from sklearn.metrics.pairwise import cosine_similarity

In [ ]:
# paths
ROOT_IMAGES = "dataset_split"
ROOT_EMBEDDINGS = "embeddings"
QUALITY_THRESHOLD = 0.20

In [ ]:
# loading MUSIQ for quality testing
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
musiq_metric = pyiqa.create_metric("musiq", device=device)
print("MUSIQ Loaded")

In [ ]:
# quality score function
def get_quality_score(image_path):
    score = musiq_metric(image_path)
    return float(score.cpu().item())

In [ ]:
# load gallery embeddings
def load_gallery_db(gallery_root):
    gallery_db = {}
    total = 0

    for identity in os.listdir(gallery_root):
        identity_dir = os.path.join(gallery_root, identity)
        gallery_db[identity] = []

        for file in os.listdir(identity_dir):
            emb = np.load(os.path.join(identity_dir, file))
            gallery_db[identity].append(emb)
            total += 1

    print("Loaded Gallery Embeddings:", total)
    return gallery_db

In [ ]:
# cosine similarity verification
def search_gallery(probe_embedding, gallery_db):
    identity_scores = {}

    for identity in gallery_db:
        sims = []

        for gallery_emb in gallery_db[identity]:
            sim = cosine_similarity(
                probe_embedding.reshape(1, -1), gallery_emb.reshape(1, -1)
            )[0][0]
            sims.append(sim)

        identity_scores[identity] = max(sims)

    sorted_scores = sorted(identity_scores.items(), key=lambda x: x[1], reverse=True)
    predicted_identity = sorted_scores[0][0]
    best_similarity = sorted_scores[0][1]
    second_similarity = sorted_scores[1][1]
    margin = best_similarity - second_similarity

    return (predicted_identity, best_similarity, second_similarity, margin)

In [ ]:
# generating dataset for training our model
def generate_dataset(split):

    gallery_root = os.path.join(ROOT_EMBEDDINGS, split, "gallery")
    probe_emb_root = os.path.join(ROOT_EMBEDDINGS, split, "degraded_probes")
    probe_img_root = os.path.join(ROOT_IMAGES, split, "degraded_probes")
    gallery_db = load_gallery_db(gallery_root)
    full_rows = []

    accepted = 0
    rejected = 0
    missing_images = 0

    counter = 0

    for identity in tqdm(os.listdir(probe_emb_root)):
        emb_dir = os.path.join(probe_emb_root, identity)
        img_dir = os.path.join(probe_img_root, identity)

        for emb_file in os.listdir(emb_dir):
            emb_path = os.path.join(emb_dir, emb_file)
            probe_embedding = np.load(emb_path)
            base_name = os.path.splitext(emb_file)[0]
            image_path = None

            for ext in [".jpg", ".jpeg", ".png"]:
                candidate = os.path.join(img_dir, base_name + ext)

                if os.path.exists(candidate):
                    image_path = candidate
                    break

            if image_path is None:
                missing_images += 1
                continue

            quality_score = get_quality_score(image_path)

            if quality_score < QUALITY_THRESHOLD:
                rejected += 1
                continue

            accepted += 1

            predicted_identity, best_similarity, second_similarity, margin = (
                search_gallery(probe_embedding, gallery_db)
            )

            label = int(predicted_identity == identity)

            full_rows.append(
                {
                    "quality_score": quality_score,
                    "best_similarity": best_similarity,
                    "second_best_similarity": second_similarity,
                    "margin": margin,
                    "true_identity": identity,
                    "predicted_identity": predicted_identity,
                    "label": label,
                }
            )

            counter += 1

            if counter % 100 == 0:
                print(f"\nProcessed: {counter}")
                print(f"Quality: {quality_score:.4f}")
                print(f"Best Similarity: {best_similarity:.4f}")
                print(f"Second Similarity: {second_similarity:.4f}")
                print(f"Margin: {margin:.4f}")
                print(f"Prediction: {predicted_identity}")
                print(f"Truth: {identity}")
                print(f"Label: {label}")

    full_df = pd.DataFrame(full_rows)
    svm_df = full_df[["quality_score", "best_similarity", "margin", "label"]]
    full_csv = os.path.join(ROOT_EMBEDDINGS, f"{split}_features_full.csv")
    svm_csv = os.path.join(ROOT_EMBEDDINGS, f"{split}_svm.csv")
    full_df.to_csv(full_csv, index=False)
    svm_df.to_csv(svm_csv, index=False)

    print(f"\nAccepted: {accepted}")
    print(f"Rejected: {rejected}")
    print(f"Missing Images: {missing_images}")
    print(f"\nSaved: {full_csv}")
    print(f"Saved: {svm_csv}")

    return full_df

In [ ]:
# generating dataset for training our model
train_df = generate_dataset("train")

In [ ]:
# stats
print("\nTRAIN DATASET")
print(train_df.shape)
print("\nLabel Counts")
print(train_df["label"].value_counts())
print("\nQuality Statistics")
print(train_df["quality_score"].describe())
print("\nMargin Statistics")
print(train_df["margin"].describe())